In [ ]:
## Setup — packages & environment

import sys, subprocess
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'networkx', 'openpyxl', 'reportlab']
for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except:
        install(pkg)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
from sklearn.decomposition import PCA
import os, datetime as dt

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Discourse Data Creation

# Synthetic discourse segments with epistemic codes
concepts = ['Knowledge', 'Reasoning', 'Community', 'Practice', 'Learning']
np.random.seed(RSEED)

# Create two groups with different discourse patterns
group1_discourse = []
group2_discourse = []

# Group 1: Heavy on knowledge and reasoning
for _ in range(40):
    pair = tuple(np.random.choice(concepts[:3], 2, replace=False))
    group1_discourse.append(pair)

# Group 2: Heavy on community and practice
for _ in range(40):
    pair = tuple(np.random.choice(concepts[2:], 2, replace=False))
    group2_discourse.append(pair)

all_discourse = group1_discourse + group2_discourse
print(f'Total discourse segments: {len(all_discourse)}')
print(f'Concepts: {concepts}')

In [ ]:
## 2. Co-occurrence Matrix

# Build co-occurrence matrix
coo_matrix = pd.DataFrame(0, index=concepts, columns=concepts)

for concept1, concept2 in all_discourse:
    coo_matrix.loc[concept1, concept2] += 1
    coo_matrix.loc[concept2, concept1] += 1  # Symmetric

print('Concept Co-occurrence Matrix:')
print(coo_matrix)

In [ ]:
## 3. ENA Network Construction

# Create network from co-occurrence
G = nx.Graph()
for i, concept1 in enumerate(concepts):
    for j, concept2 in enumerate(concepts):
        if i < j and coo_matrix.loc[concept1, concept2] > 0:
            G.add_edge(concept1, concept2, weight=coo_matrix.loc[concept1, concept2])

# Add isolated nodes
for concept in concepts:
    if concept not in G:
        G.add_node(concept)

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
## 4. Visualizations

os.makedirs('figures', exist_ok=True)

# 1. Co-occurrence heatmap
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(coo_matrix, annot=True, fmt='d', cmap='YlOrRd', square=True, ax=ax)
ax.set_title('Concept Co-occurrence Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/01_cooccurrence_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_cooccurrence_matrix.png')

# 2. Epistemic network
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G, k=2, iterations=50, seed=RSEED)

# Node size based on degree
node_sizes = [500 * G.degree(n) for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightblue', ax=ax, alpha=0.8)

# Edge width based on weight
edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]
max_weight = max(weights) if weights else 1
edge_widths = [3 * w / max_weight for w in weights]

nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)

ax.set_title('Epistemic Network (ENA)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/02_epistemic_network.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_epistemic_network.png')

# 3. Node degree comparison
fig, ax = plt.subplots(figsize=(9, 6))
degrees = {n: G.degree(n) for n in G.nodes()}
colors = sns.color_palette('Set2', len(concepts))
ax.bar(range(len(degrees)), list(degrees.values()), color=colors, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(degrees)))
ax.set_xticklabels(list(degrees.keys()), rotation=45)
ax.set_ylabel('Degree Centrality', fontsize=12)
ax.set_title('Concept Connectivity in Discourse', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('figures/03_concept_centrality.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_concept_centrality.png')

In [ ]:
## 5. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch18_ENA_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 18: Epistemic Network Analysis</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Epistemic Network Analysis (ENA) visualizes knowledge and discourse patterns. '
    'Co-occurrence of epistemic codes reveals cognitive structures and disciplinary thinking patterns.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Methods</b>', styles['Heading2']))
methods = (
    f'Discourse segments coded with concepts: {', '.join(concepts)}. '
    'Co-occurrence matrix computed. Network visualized with concept centrality.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/02_epistemic_network.png'):
        story.append(Image('figures/02_epistemic_network.png', width=500, height=400))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Epistemic network showing concept relationships.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>4. Interpretation</b>', styles['Heading2']))
interp = (
    'Network structure reveals conceptual relationships in student discourse. '
    'Dense clusters indicate tightly related concepts; isolated nodes may indicate underdeveloped understanding.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')